# 5. Exercise: from ε to a turbulent flux

We use VMP data from PIRATA FR-36 (March 2026) at two stations along 10°W:

- **9.5°S (station 26)**: away from the equator, where currents are weak;
- **0°N (station 7)**: on the equator, where the Equatorial Undercurrent flows below the surface.

We follow the data as an operator does:

1. **The profile comes out of the water**: what do we check in the raw data?
2. **From L1 to $\varepsilon$**: the processing, in short.
3. **Can we trust $\varepsilon$?** The spectra, the two probes and the quality checks.
4. **What does the water column tell us?** The profiles, and the Richardson number from conventional measurements (CTD and ADCP).
5. **From $\varepsilon$ to a flux.** The eddy diffusivity and the heat flux below the mixed layer.

**How it works.** Run the cells one by one, from the top (Shift + Enter). The cells marked **✏️ Your turn** are yours. Below each of them, a hidden cell **Solution** holds a solution: try first, and if you are stuck after 3–4 minutes, double-click on it to show it, and run it. We discuss the questions together.

**Setup.** In Google Colab, run the cell below first.

In [ ]:
# Setup: run this cell first (Colab only). It copies the course data and installs gsw.
import sys, os
if "google.colab" in sys.modules:
    repo = "/content/catching-ocean-turbulence"
    if os.path.exists(repo):
        !git -C {repo} pull -q                      # already copied: get the latest files
    else:
        !git clone -q https://github.com/mgentil17/catching-ocean-turbulence.git {repo}
    os.chdir(repo)
    !pip install -q gsw

## Step 1. The profile comes out of the water

Right after a profile, before any processing, we check the raw data (part 4, the quick check on board with Zissou). For each station, the file holds the descent of the VMP, sampled 512 times per second:

- the pressure and the fall speed $W$;
- the shear of the two probes, $\partial u/\partial z$ in s⁻¹: the signals are already converted to physical units (ATOMIX level L1);
- the accelerometers $A_x$ and $A_y$, in counts: the vibrations of the instrument.

In [ ]:
# Load the libraries and open the file of VMP profiles
import numpy as np
import xarray as xr
import gsw
import matplotlib.pyplot as plt

prof = xr.open_dataset("data/fr36_profiles.nc")
prof

In [ ]:
# Open the raw (L1) files of the two stations and look at what they contain
l1 = {s: xr.open_dataset(f"data/fr36_L1_{s}.nc") for s in ["stat26", "stat7"]}
name = {"stat26": "9.5°S (station 26)", "stat7": "0°N (station 7)"}
l1["stat7"]

In [ ]:
# Quick check: pressure and fall speed, shear of the two probes, accelerometers, against time
def quicklook(s, t1=None, t2=None):
    d = l1[s]
    t = d.time.values - d.time.values[0]                  # seconds from the start of the descent
    sel = slice(None) if t1 is None else (t >= t1) & (t <= t2)
    fig, axs = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    axs[0].plot(t[sel], d.pres.values[sel], color="k")
    axs[0].invert_yaxis(); axs[0].set_ylabel("pressure (dbar)")
    aw = axs[0].twinx()
    aw.plot(t[sel], d.W.values[sel], color="tab:red"); aw.set_ylabel("W (m s⁻¹)", color="tab:red")
    off = 1.0 if t1 is not None else 2.0                  # shift of probe 2, for clarity
    axs[1].plot(t[sel], d.shear.sel(probe=1).values[sel], color="tab:blue", lw=0.5, label="probe 1")
    axs[1].plot(t[sel], d.shear.sel(probe=2).values[sel] + off, color="tab:orange", lw=0.5, label=f"probe 2 (+{off:g})")
    axs[1].set_ylabel("∂u/∂z (s⁻¹)"); axs[1].legend(loc="upper right", fontsize=8)
    if t1 is None:
        axs[1].set_ylim(-1.5, 3.5)
    axs[2].plot(t[sel], d.acc.sel(axis="Ax").values[sel], color="tab:blue", lw=0.5, label="Ax")
    axs[2].plot(t[sel], d.acc.sel(axis="Ay").values[sel] + 500, color="tab:orange", lw=0.5, label="Ay (+500)")
    axs[2].set_ylabel("acceleration (counts)"); axs[2].legend(loc="upper right", fontsize=8)
    axs[2].set_xlabel("time from the start of the descent (s)")
    for ax in axs:
        ax.grid(alpha=0.3)
    fig.suptitle(name[s])

quicklook("stat26")
quicklook("stat7")

### ❓ Questions 1.1 and 1.2

**1.1** Look at the red line, the fall speed $W$. Does the VMP fall at a steady speed (about 0.7 m s⁻¹)?

**1.2** Look at the shear of the two probes. Where is the signal strong? Is it strong on both probes at the same time?

**✏️ Your turn.** At the equator, zoom on two events of the shear signal, with `quicklook("stat7", t1, t2)`: one between 20 and 40 s, and one between 67 and 69 s.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title Solution (double-click to show)

quicklook("stat7", 20, 40)
quicklook("stat7", 67, 69)

### ❓ Question 1.3

One zoom shows turbulence, the other an artefact (something hit one probe). Which is which, and why?

## Step 2. From L1 to ε: the processing

The processing follows ATOMIX (Lueck et al., 2024; part 4), with the toolbox of C. Vic:

- **L1, converted**: the signals in physical units (step 1);
- **L2, cleaned**: the impacts removed, and the part of the shear caused by the vibrations subtracted;
- **L3, spectra**: the shear spectrum of each probe, in segments of a few meters;
- **L4, dissipation**: $\varepsilon$ from the spectra, with its quality checks.

The next step works with L3 and L4: is the final $\varepsilon$ good?

## Step 3. Can we trust ε?

For each station, a second file gives the details of the turbulence measurement. The descent is cut into **segments** of a few meters. For each segment, the shear spectrum of each probe is integrated to give $\varepsilon$ (part 4): $\varepsilon_1$ and $\varepsilon_2$ for the two probes, and a final $\varepsilon$ after the quality checks.

In [ ]:
stations = {"stat26": "tab:blue", "stat7": "tab:red"}     # 9.5°S, 0°N
# Open the segment files of the two stations and look at what they contain
seg = {s: xr.open_dataset(f"data/fr36_segments_{s}.nc") for s in stations}
seg["stat7"]

In [ ]:
# Plot eps of each probe and the final eps along the descent, at the two stations
fig, axs = plt.subplots(1, 2, figsize=(9, 5), sharey=True, sharex=True)
for ax, s in zip(axs, stations):
    d = seg[s]
    o = np.argsort(d.seg_pres.values)
    ax.semilogx(d.seg_eps_probe.sel(probe=1), d.seg_pres, ".", ms=3, color="tab:blue", label="ε₁")
    ax.semilogx(d.seg_eps_probe.sel(probe=2), d.seg_pres, ".", ms=3, color="tab:red", label="ε₂")
    ax.semilogx(d.seg_eps_final.values[o], d.seg_pres.values[o], "k-", lw=1, label="ε final")
    ax.set_title(f"{seg[s].attrs['lat']:.1f}°N"); ax.set_xlabel("ε (W kg⁻¹)"); ax.grid(alpha=0.3)
axs[0].set_xlim(1e-11, 1e-5); axs[0].set_ylim(200, 0); axs[0].set_ylabel("pressure (dbar)"); axs[0].legend(fontsize=8)
fig.suptitle("Dissipation rate. Dashed: base of the mixed layer");

Do the two probes (blue and red dots) follow each other along the profile? The black line is the final $\varepsilon$: where a segment fails the quality checks, it has a gap. We look closer at the spectra.

**✏️ Your turn.** On the equatorial profile, choose the pressure of one segment with a high $\varepsilon$ and one with a low $\varepsilon$.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title Solution (double-click to show)
# Pressures (dbar) of one segment with high eps and one with low eps, at the equator
p_high, p_low = 40, 130

In [ ]:
# Model spectrum of Nasmyth (fit of Wolk et al., 2002), normalised so that 15/2 nu * integral = eps
def model_spectrum(k_cpm, eps, nu=1e-6):
    ks = (eps / nu**3) ** 0.25                      # Kolmogorov wavenumber (cpm)
    G = lambda x: 8.05 * x**(1 / 3) / (1 + (20.6 * x) ** 3.715)
    xx = np.logspace(-5, 2, 4000)
    trapz = getattr(np, "trapezoid", None) or np.trapz
    norm = 2 * eps / (15 * nu) / (ks * trapz(G(xx), xx))
    return norm * G(k_cpm / ks)

In [ ]:
# Plot the measured spectra of the two probes and the model spectrum, for the two chosen segments
d = seg["stat7"]
fig, ax = plt.subplots(figsize=(6, 4.5))
for p0, lab in [(p_high, "high ε"), (p_low, "low ε")]:
    i = int(np.argmin(np.abs(d.seg_pres.values - p0)))       # closest segment
    k, e = d.seg_k.values[i], float(d.seg_eps_final[i])
    ax.loglog(k, d.seg_spec.sel(probe=1).values[i], color="tab:blue", lw=1)
    ax.loglog(k, d.seg_spec.sel(probe=2).values[i], color="tab:red", lw=1)
    ax.loglog(k, model_spectrum(k, e), color="0.4", lw=2.5)
    ax.text(1, 2 * model_spectrum(1.0, e), f"{lab}: ε = {e:.1e} W kg⁻¹, {d.seg_pres.values[i]:.0f} dbar", fontsize=8)
ax.set_xlim(0.5, 300); ax.set_ylim(1e-9, 1e-2)
ax.set_xlabel("k (cpm)"); ax.set_ylabel("Ψ (s⁻² cpm⁻¹)")
ax.set_title("Equator: probe 1 (blue), probe 2 (red), model (grey)");

### ❓ Question 3.1

How does the spectrum change from the low-$\varepsilon$ to the high-$\varepsilon$ segment? Over which range of wavenumbers do the measured spectra follow the model, and what happens beyond?

**✏️ Your turn.** Two probes a few centimetres apart see the same turbulence, so their $\varepsilon$ should agree within a factor of 2, that is $1/2 < \varepsilon_2/\varepsilon_1 < 2$.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title Solution (double-click to show)
# Fraction of segments where the two probes agree within a factor of 2
for s in stations:
    e1 = seg[s].seg_eps_probe.sel(probe=1).values      # eps of probe 1
    e2 = seg[s].seg_eps_probe.sel(probe=2).values      # eps of probe 2
    ratio = e2 / e1
    good = (ratio > 0.5) & (ratio < 2)                 # True where 1/2 < ratio < 2
    fraction = np.mean(good)
    print(seg[s].attrs["lat"], "°N:", round(100 * fraction), "% of the segments agree")

**Quality checks (reminder, part 4).** In the processing (toolbox of Vic et al., following ATOMIX; Lueck et al., 2024), each estimate of $\varepsilon$ is checked:

- **the shape of the spectrum**: the measured spectrum must follow the model spectrum (part 4);
- **the resolved part**: most of the spectrum must be measured above the noise and within the range the probe resolves;
- **the instrument**: the VMP must fall steadily, without strong tilt or vibrations.

The final $\varepsilon$ is the mean of the two probes when both pass the checks, the value of the only probe that passes, or nothing when neither passes.

In [ ]:
# Plot eps of probe 2 against eps of probe 1, and show what the quality checks did with each segment
fig, ax = plt.subplots(figsize=(5, 5))
x = np.logspace(-10, -5, 50)
ax.fill_between(x, x / 2, 2 * x, color="0.85", label="within a factor of 2")
ax.plot(x, x, "k-", lw=1)
for s, c in stations.items():
    e1, e2 = seg[s].seg_eps_probe.sel(probe=1).values, seg[s].seg_eps_probe.sel(probe=2).values
    kept = np.isfinite(seg[s].seg_eps_final.values)
    agree = (e2 / e1 > 0.5) & (e2 / e1 < 2)
    lat = f"{seg[s].attrs['lat']:.1f}°N"
    ax.loglog(e1[kept & agree], e2[kept & agree], "o", ms=4, mfc="none", color=c, label=f"{lat}: kept, probes agree")
    ax.loglog(e1[kept & ~agree], e2[kept & ~agree], "^", ms=5, color=c, label=f"{lat}: kept, one probe only")
    ax.loglog(e1[~kept], e2[~kept], "x", ms=5, color=c, label=f"{lat}: rejected")
    print(f"{lat}: {kept.sum()} kept ({(kept & ~agree).sum()} from one probe only), {(~kept).sum()} rejected")
ax.set_xlim(1e-10, 1e-5); ax.set_ylim(1e-10, 1e-5); ax.set_aspect("equal")
ax.set_xlabel("ε₁ (W kg⁻¹)"); ax.set_ylabel("ε₂ (W kg⁻¹)"); ax.legend(fontsize=6, loc="upper left");

### ❓ Question 3.2

Do the two probes measure the same turbulence? What did the quality checks do with the segments where the probes disagree, and with the strongest $\varepsilon$ at the equator?

## Step 4. What the water column tells us

We now use $\varepsilon$ with the other measurements of the station: temperature, salinity and oxygen of the VMP, and the current of the L-ADCP. The base of the mixed layer (MLD: density criterion of 0.03 kg m⁻³, de Boyer Montégut et al., 2004) is computed for you: it is our reference depth in steps 4 and 5.

In [ ]:
# For our two stations, compute the potential density and the stratification N2 from temperature and salinity
pres = prof.pres.values
stations = {"stat26": "tab:blue", "stat7": "tab:red"}     # 9.5°S, 0°N

col = {}
for s in stations:
    p = prof.sel(station=s)
    SA = gsw.SA_from_SP(p.sal.values, pres, float(p.lon), float(p.lat))
    CT = gsw.CT_from_t(SA, p.temp.values, pres)
    n2, p_mid = gsw.Nsquared(SA, CT, pres, lat=float(p.lat))
    col[s] = {"temp": p.temp.values, "o2": p.o2.values, "sigma": gsw.sigma0(SA, CT),
              "N2": np.interp(pres, p_mid, n2), "lat": float(p.lat)}

In [ ]:
# MLD: first depth where the density exceeds the shallowest density by 0.03 kg m-3
for s in stations:
    sigma = col[s]["sigma"]
    ref = sigma[np.isfinite(sigma)][0]              # density of the shallowest measurement
    col[s]["mld"] = pres[np.argmax(sigma > ref + 0.03)]
    print(col[s]["lat"], "°N: MLD =", col[s]["mld"], "m")

The currents were measured by a lowered ADCP (L-ADCP), fixed on the CTD frame, during the CTD cast made at the same station just before the VMP. They are averaged over 16 m. Open the file and look at what it contains.

In [ ]:
# Open the file of currents (L-ADCP) and stratification (CTD) at the two stations
ss = xr.open_dataset("data/shear_strat_stat7_stat26.nc").drop_vars("Ri")
ss

In [ ]:
# Plot temperature, stratification, oxygen, eastward current and eps at the two stations
fig, axs = plt.subplots(1, 5, figsize=(15, 4.5), sharey=True)
for s, c in stations.items():
    lab = f"{col[s]['lat']:.1f}°N"
    axs[0].plot(col[s]["temp"], pres, color=c, label=lab)
    axs[1].plot(col[s]["N2"], pres, color=c)
    axs[2].plot(col[s]["o2"], pres, color=c)
    axs[3].plot(ss.u.sel(station=s), ss.depth, "o-", ms=3, color=c)
    axs[4].semilogx(prof.eps.sel(station=s), pres, color=c, lw=1)
    for ax in axs:
        ax.axhline(col[s]["mld"], color=c, ls="--", lw=1)
axs[3].axvline(0, color="k", lw=0.8)
labels = ["T (°C)", "N² (s⁻²)", "O₂ (µmol kg⁻¹)", "u, eastward (m s⁻¹)", "ε (W kg⁻¹)"]
for ax, label in zip(axs, labels):
    ax.set_xlabel(label); ax.grid(alpha=0.3)
axs[4].set_xlim(1e-10, 1e-6)
axs[0].set_ylim(200, 0); axs[0].set_ylabel("depth (m)"); axs[0].legend()
fig.suptitle("Two profiles. Dashed: base of the mixed layer (MLD)");

$\varepsilon$ needs a microstructure profiler. The **shear** of the current, $S^2 = (\partial u/\partial z)^2 + (\partial v/\partial z)^2$, and the **stratification** $N^2$ come from conventional measurements: the L-ADCP and the CTD. The shear feeds turbulence, the stratification resists it. Their ratio is the **gradient Richardson number**:

$$
Ri = \frac{N^2}{S^2}
$$

When $Ri < 1/4$, the shear can overcome the stratification and the flow can become unstable (Miles, 1961; Howard, 1961).

$S^2$ and $N^2$ are computed on the 16-m grid of the L-ADCP (file `ss`). We compare $Ri$ with the measured $\varepsilon$ in two layers below the mixed layer: the first 40 m (three levels of the grid), and from 40 to 150 m.

In [ ]:
# Average eps of the VMP on the 16-m grid of S2 and N2
zm = ss.depth_mid.values                     # depths of the 16-m grid where S2, N2 and Ri are computed
eps16 = {}
for s in stations:
    eps16[s] = np.array([np.nanmean(prof.eps.sel(station=s).values[(pres >= z - 8) & (pres < z + 8)]) for z in zm])

**✏️ Your turn.** Compute $Ri$ from `ss.N2` and `ss.S2`, and select the first 40 m below the mixed layer.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title Solution (double-click to show)
# Gradient Richardson number, and selection of the first 40 m below the mixed layer
Ri = ss.N2 / ss.S2                            # Ri = N2 / S2
layer = {}
for s in stations:
    mld = col[s]["mld"]
    layer[s] = (zm > mld) & (zm <= mld + 40)  # True where mld < zm <= mld + 40

In [ ]:
# Current, shear, stratification, Ri and eps against the depth below the mixed layer (MLD + dz), as in Hummels et al. (2013)
fig, axs = plt.subplots(1, 5, figsize=(16, 5), sharey=True)
for s, c in stations.items():
    lab = f"{col[s]['lat']:.1f}°N"
    dz_u = ss.depth.values - col[s]["mld"]          # depth below the mixed layer, for u
    dz_m = zm - col[s]["mld"]                       # depth below the mixed layer, for S2, N2, Ri and eps
    axs[0].plot(ss.u.sel(station=s), dz_u, "o-", ms=3, color=c, label=lab)
    axs[1].semilogx(ss.S2.sel(station=s), dz_m, "o-", ms=3, color=c)
    axs[2].semilogx(ss.N2.sel(station=s), dz_m, "o-", ms=3, color=c)
    axs[3].semilogx(Ri.sel(station=s), dz_m, "o-", ms=3, color=c)
    axs[4].semilogx(prof.eps.sel(station=s), pres - col[s]["mld"], color=c, lw=0.5, alpha=0.4)
    axs[4].semilogx(eps16[s], dz_m, "o-", ms=3, color=c)
axs[0].axvline(0, color="k", lw=0.8)
axs[3].axvline(0.25, color="k", ls="--", lw=1, label="Ri = 1/4")
axs[3].axvline(1, color="k", ls=":", lw=1, label="Ri = 1")
labels = ["u, eastward (m s⁻¹)", "S², shear (s⁻²)", "N², stratification (s⁻²)", "Ri", "ε (W kg⁻¹)"]
for ax, label in zip(axs, labels):
    ax.axhspan(-50, 0, color="0.85")            # mixed layer
    ax.axhspan(40, 150, color="0.85")           # deeper than 40 m below the mixed layer
    ax.axhline(0, color="k", lw=1)
    ax.set_xlabel(label); ax.grid(alpha=0.3)
axs[1].set_xlim(1e-7, 3e-3); axs[2].set_xlim(1e-7, 3e-3); axs[3].set_xlim(0.05, 100); axs[4].set_xlim(1e-9, 1e-6)
axs[0].set_ylim(150, -50); axs[0].set_ylabel("MLD + dz (m)")
axs[0].legend(); axs[3].legend(fontsize=7)
fig.suptitle("0 = base of the mixed layer. White: the first 40 m below it");

The cell below gives the median of $\varepsilon$, $S^2$, $N^2$ and $Ri$ in the two layers.

In [ ]:
# Median of eps, S2, N2 and Ri on the 16-m grid, 0-40 m and 40-150 m below the mixed layer
for s in stations:
    mld = col[s]["mld"]
    eps = eps16[s]
    S2 = ss.S2.sel(station=s).values
    N2 = ss.N2.sel(station=s).values
    R = Ri.sel(station=s).values

    below = (zm > mld) & (zm <= mld + 40)           # 0 to 40 m below the mixed layer
    deeper = (zm > mld + 40) & (zm <= mld + 150)    # 40 to 150 m below the mixed layer

    print(col[s]["lat"], "°N")
    print("  below : eps =", f"{np.median(eps[below]):.1e}")
    print("          S2  =", f"{np.median(S2[below]):.1e}")
    print("          N2  =", f"{np.median(N2[below]):.1e}")
    print("          Ri  =", f"{np.median(R[below]):.1e}")
    print("  deeper: eps =", f"{np.median(eps[deeper]):.1e}")
    print("          S2  =", f"{np.median(S2[deeper]):.1e}")
    print("          N2  =", f"{np.median(N2[deeper]):.1e}")
    print("          Ri  =", f"{np.median(R[deeper]):.1e}")

### ❓ Question 4.1

Compare $Ri$ and $\varepsilon$ at the two stations, in the two layers. Does a lower $Ri$ always mean a larger $\varepsilon$? What does $Ri$ tell us, and what does $\varepsilon$ tell us?

## Step 5. From $\varepsilon$ to a heat flux

This is the end of the chain (part 2). First the eddy diffusivity, following Osborn (1980):

$$
K_\rho = \Gamma\,\frac{\varepsilon}{N^2}, \qquad \Gamma = 0.2
$$

then the heat flux, with $z$ positive downward (a positive flux goes **down**, out of the surface layer):

$$
F_{heat} = -\rho_0\,c_p\,K_\rho\,\frac{\partial T}{\partial z} \quad (\text{W m}^{-2})
$$

The cell below computes the vertical gradient of temperature on the same 16-m grid as $N^2$ and $\varepsilon$.

In [ ]:
# Gradient of temperature (CTD) on the 16-m grid, at the same depths zm as eps16 and N2
GAMMA, RHO0, CP = 0.2, 1025.0, 3990.0     # mixing efficiency, density (kg m-3), heat capacity (J kg-1 K-1)
z = ss.depth.values                       # centres of the 16-m layers
dTdz = {}
for s in stations:
    T16 = ss.temp.sel(station=s).values   # CTD, 16-m means
    dTdz[s] = np.diff(T16) / np.diff(z)   # °C per m, at the depths zm

**✏️ Your turn.** Compute $K_\rho$ and the heat flux at each station, on the 16-m grid. Keep only the stratified water below the mixed layer, where Osborn's model applies.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title Solution (double-click to show)

# Eddy diffusivity and heat flux below the mixed layer, on the 16-m grid
K, F_heat = {}, {}
for s in stations:
    N2 = ss.N2.sel(station=s).values
    ok = (zm > col[s]["mld"]) & (N2 > 0)              # stratified water below the mixed layer
    K[s] = np.where(ok, GAMMA * eps16[s] / N2, np.nan)
    F_heat[s] = -RHO0 * CP * K[s] * dTdz[s]

In [ ]:
# K, temperature gradient and heat flux against the depth below the mixed layer; white: the first 40 m below it
fig, axs = plt.subplots(1, 3, figsize=(12, 5), sharey=True)
for s, c in stations.items():
    dz = zm - col[s]["mld"]                          # depth below the mixed layer
    lab = f"{col[s]['lat']:.1f}°N"
    axs[0].semilogx(K[s], dz, "o-", ms=3, color=c, label=lab)
    axs[1].plot(dTdz[s], dz, "o-", ms=3, color=c)
    axs[2].plot(F_heat[s], dz, "o-", ms=3, color=c)
axs[0].axvline(1e-5, color="tab:green", ls="--", lw=1, label="ocean interior (Ledwell et al., 1993)")
axs[0].axvline(1.4e-7, color="tab:purple", ls=":", lw=1, label="molecular (heat)")
labels = ["K_ρ (m² s⁻¹)", "dT/dz (°C m⁻¹)", "heat flux (W m⁻²)"]
for ax, label in zip(axs, labels):
    ax.axhspan(-50, 0, color="0.85")            # mixed layer
    ax.axhspan(40, 150, color="0.85")           # deeper than 40 m below the mixed layer
    ax.axhline(0, color="k", lw=1)
    ax.set_xlabel(label); ax.grid(alpha=0.3)
for ax in axs[1:]:
    ax.axvline(0, color="k", lw=0.8)
axs[0].set_xlim(1e-7, 1e-3)
axs[0].set_ylim(150, -50); axs[0].set_ylabel("MLD + dz (m)"); axs[0].legend(fontsize=7)
fig.suptitle("0 = base of the mixed layer. White: the first 40 m below it");

The cell below gives the median of $K_\rho$ and of the heat flux in the two layers.

In [ ]:
# Median of K and heat flux, 0-40 m and 40-150 m below the mixed layer
for s in stations:
    mld = col[s]["mld"]
    below = (zm > mld) & (zm <= mld + 40)           # 0 to 40 m below the mixed layer
    deeper = (zm > mld + 40) & (zm <= mld + 150)    # 40 to 150 m below the mixed layer
    print(col[s]["lat"], "°N")
    print("  below : K      =", f"{np.nanmedian(K[s][below]):.1e}", "m2 s-1")
    print("          F_heat =", f"{np.nanmedian(F_heat[s][below]):.1f}", "W m-2")
    print("  deeper: K      =", f"{np.nanmedian(K[s][deeper]):.1e}", "m2 s-1")
    print("          F_heat =", f"{np.nanmedian(F_heat[s][deeper]):.1f}", "W m-2")

### ❓ Questions 5.1 and 5.2

**5.1** Compare $K_\rho$ at the two stations with the molecular diffusivity of heat ($1.4 \times 10^{-7}$ m² s⁻¹) and with the value of the ocean interior ($10^{-5}$ m² s⁻¹, Ledwell et al., 1993).

**5.2** Just below the mixed layer, which station removes more heat from the surface layer? Does the difference come from $K_\rho$ or from the temperature gradient?

## To go further

In step 4, we compared $Ri$, computed from conventional measurements (CTD and ADCP), with $\varepsilon$, measured by the VMP. **Parameterizations** go one step further: they estimate $\varepsilon$, and so the fluxes, from CTD and ADCP data alone. Microstructure measurements like these are what we need to build and test them.